# Cognifyz Technologies - Data Analysis Internship
## Level 3 Analysis
**Candidate Name:** Jeevani Vedavendra  
**Reference ID:** CTI/A1/C421603  
**Domain:** Data Analysis  

---
### Overview of Level 3 Tasks:
1. **Task 1: Restaurant Reviews** - Analyze text reviews to identify common positive and negative keywords, calculate average review length, and examine the relationship with ratings.
2. **Task 2: Votes Analysis** - Identify restaurants with the highest and lowest votes and analyze the correlation between votes and rating.
3. **Task 3: Price Range vs. Online Delivery and Table Booking** - Analyze the relationship between price range and service availability.


### Step 1: Import Libraries and Load Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'Segoe UI', 'font.size': 11})

df = pd.read_csv('../Dataset .csv')
print(f"Dataset Shape: {df.shape}")
df.head(3)


---
## Task 1: Restaurant Reviews
- Analyze text reviews to identify the most common positive and negative keywords.
- Calculate the average length of reviews and explore if there is a relationship between review length and rating.


In [ ]:
# Text length metrics
df['Review_Text'] = df['Rating text'].astype(str)
df['Review_Char_Len'] = df['Review_Text'].apply(len)
df['Review_Word_Count'] = df['Review_Text'].apply(lambda x: len(x.split()))

print(f"Average Review Character Length: {df['Review_Char_Len'].mean():.2f}")
print(f"Average Review Word Count: {df['Review_Word_Count'].mean():.2f}")

# Keyword frequency analysis
words_series = df['Review_Text'].str.lower().str.split().explode()
word_counts = words_series.value_counts()

print("\nKey Sentiment Words:")
for word in ['excellent', 'very', 'good', 'average', 'poor', 'not']:
    print(f"- '{word}': {word_counts.get(word, 0):,} occurrences")

# Grouped breakdown
review_summary = df.groupby('Rating text').agg(
    Count=('Aggregate rating', 'count'),
    Avg_Rating=('Aggregate rating', 'mean'),
    Avg_Char_Len=('Review_Char_Len', 'mean')
).sort_values(by='Avg_Rating', ascending=False)
display(review_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

rev_cat_df = review_summary.reset_index()
sns.barplot(data=rev_cat_df, x='Rating text', y='Count', palette='Blues_r', hue='Rating text', legend=False, ax=axes[0])
axes[0].set_title("Frequency of Review Rating Categories", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Review Category")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=df, x='Rating text', y='Aggregate rating', order=['Excellent', 'Very Good', 'Good', 'Average', 'Poor', 'Not rated'], palette='Spectral', hue='Rating text', legend=False, ax=axes[1])
axes[1].set_title("Rating Distribution Across Categories", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Review Category")
axes[1].set_ylabel("Aggregate Rating (0 - 5)")
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


**Key Findings - Task 1:**
- Positive sentiment terms (**Good**: 3,179, **Very Good**: 1,079, **Excellent**: 301) collectively characterize 47.7% of all restaurants.
- Average sentiment is the single largest category with 3,737 restaurants (39.1%).
- Unrated restaurants account for 2,148 records with 'Not rated' text.


---
## Task 2: Votes Analysis
- Identify the restaurants with the highest and lowest number of votes.
- Analyze if there is a correlation between the number of votes and the rating of a restaurant.


In [ ]:
max_votes_val = df['Votes'].max()
min_votes_val = df['Votes'].min()

top_voted = df[df['Votes'] == max_votes_val][['Restaurant Name', 'City', 'Votes', 'Aggregate rating']]
print(f"Restaurant with highest votes: {top_voted.iloc[0]['Restaurant Name']} ({top_voted.iloc[0]['Votes']:,} votes)")

zero_votes_count = (df['Votes'] == min_votes_val).sum()
print(f"Restaurants with lowest votes (0 votes): {zero_votes_count} restaurants")

# Correlations
corr_all = df['Votes'].corr(df['Aggregate rating'])
spearman_corr, _ = stats.spearmanr(df['Votes'], df['Aggregate rating'])
rated_df = df[df['Aggregate rating'] > 0]
corr_rated = rated_df['Votes'].corr(rated_df['Aggregate rating'])

print(f"Pearson Correlation (All Data): {corr_all:.4f}")
print(f"Spearman Rank Correlation: {spearman_corr:.4f}")
print(f"Pearson Correlation (Rated Outlets Only): {corr_rated:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter with regression
sns.regplot(data=df, x='Votes', y='Aggregate rating', scatter_kws={'alpha': 0.3, 'color': '#2a9d8f'}, line_kws={'color': '#e76f51', 'linewidth': 2}, ax=axes[0])
axes[0].set_title(f"Votes vs. Rating (r = {corr_all:.2f})", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Number of Votes")
axes[0].set_ylabel("Aggregate Rating (0 - 5)")

# Log-scaled votes
sns.regplot(data=rated_df, x=np.log1p(rated_df['Votes']), y='Aggregate rating', scatter_kws={'alpha': 0.3, 'color': '#457b9d'}, line_kws={'color': '#e63946', 'linewidth': 2}, ax=axes[1])
axes[1].set_title("Log(Votes + 1) vs. Rating (Rated Restaurants)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Log(Votes + 1)")
axes[1].set_ylabel("Aggregate Rating")

plt.tight_layout()
plt.show()


**Key Findings - Task 2:**
- **Toit** (Bangalore) holds the highest number of votes (**10,934 votes**) with an outstanding rating of **4.8**.
- There is a **statistically significant positive correlation (r = 0.31 overall, r = 0.41 for rated restaurants, Spearman rho = 0.85)** between vote counts and aggregate ratings. More popular, highly patronized restaurants systematically maintain higher customer satisfaction.


---
## Task 3: Price Range vs. Online Delivery and Table Booking
- Analyze if there is a relationship between the price range and the availability of online delivery and table booking.
- Determine if higher-priced restaurants are more likely to offer these services.


In [ ]:
delivery_crosstab = pd.crosstab(df['Price range'], df['Has Online delivery'], normalize='index') * 100
booking_crosstab = pd.crosstab(df['Price range'], df['Has Table booking'], normalize='index') * 100

summary_services = pd.DataFrame({
    'Price_Range': [1, 2, 3, 4],
    'Online_Delivery_Pct': [delivery_crosstab.loc[pr, 'Yes'] for pr in [1, 2, 3, 4]],
    'Table_Booking_Pct': [booking_crosstab.loc[pr, 'Yes'] for pr in [1, 2, 3, 4]]
})

display(summary_services)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Grouped bar chart
x = np.arange(len(summary_services))
width = 0.35
axes[0].bar(x - width/2, summary_services['Online_Delivery_Pct'], width, label='Online Delivery', color='#2a9d8f')
axes[0].bar(x + width/2, summary_services['Table_Booking_Pct'], width, label='Table Booking', color='#e76f51')
axes[0].set_title("Service Availability by Price Range", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Price Range (1 = Budget, 4 = Fine Dining)")
axes[0].set_ylabel("Percentage (%)")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"Tier {pr}" for pr in summary_services['Price_Range']])
axes[0].legend()

# 100% stacked bar chart for Table Booking
booking_pct_df = booking_crosstab.reset_index()
axes[1].bar(booking_pct_df['Price range'], booking_pct_df['Yes'], color='#e76f51', label='Has Table Booking', alpha=0.9)
axes[1].bar(booking_pct_df['Price range'], booking_pct_df['No'], bottom=booking_pct_df['Yes'], color='#adb5bd', label='No Table Booking', alpha=0.6)
axes[1].set_title("Table Booking Availability Across Price Ranges (Stacked)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Price Range Tier")
axes[1].set_ylabel("Percentage (%)")
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()


**Key Findings - Task 3:**
- **Table Booking:** Strongly correlates with price tier. Only **0.02%** of budget restaurants (Tier 1) offer booking, compared to **45.74%** in Tier 3 and **46.76%** in Tier 4.
- **Online Delivery:** Peaks in **Tier 2 (41.31%)** and drops significantly in luxury Tier 4 restaurants (**9.04%**), reflecting the fine-dining emphasis on on-premise experiential dining over takeout.
